# Project 23: Student Performance Prediction and Early Intervention
This notebook covers Stages 1-8 of the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import lime
import lime.lime_tabular
import joblib
import os

os.makedirs("results", exist_ok=True)

# ==========================================
# STAGES 1 & 2: Data Collection & Definition
# ==========================================
from ucimlrepo import fetch_ucirepo
print("Fetching Dataset...")
student_performance = fetch_ucirepo(id=320)
X = student_performance.data.features
y = student_performance.data.targets

df = pd.concat([X, y], axis=1)

df['Target_Pass'] = (df['G3'] >= 10).astype(int)
df_clean = df.drop(columns=['G1', 'G2', 'G3'])

# ==========================================
# STAGE 3: Data Preprocessing
# ==========================================
numeric_features = df_clean.drop(columns=['Target_Pass']).select_dtypes(include=np.number).columns.tolist()
categorical_features = df_clean.drop(columns=['Target_Pass']).select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])

# ==========================================
# STAGE 4: Exploratory Data Analysis (EDA)
# ==========================================
plt.figure(figsize=(6,4))
sns.countplot(data=df_clean, x='Target_Pass', palette="Set2")
plt.title("Class Distribution (0 = Fail, 1 = Pass)")
plt.savefig("results/class_distribution.png")
plt.close()
print("EDA Plot saved to results/class_distribution.png")

# ==========================================
# STAGES 5, 6 & 7: Feature Eng, Modeling, Evaluation
# ==========================================
X_model = df_clean.drop('Target_Pass', axis=1)
y_model = df_clean['Target_Pass']

X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42, stratify=y_model)

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

best_model = None
best_auc = 0
best_name = ""

print("\nTraining Models...")
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred))
    print(f"ROC AUC: {auc:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        best_model = pipeline
        best_name = name

print(f"\nBest Model: {best_name} (AUC: {best_auc:.4f})")
joblib.dump(best_model, 'results/best_student_model.joblib')

# ==========================================
# STAGE 8: Model Interpretability (LIME)
# ==========================================
print("\nGenerating LIME Explanation for an At-Risk Student...")
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

X_train_dense = X_train_transformed.toarray() if hasattr(X_train_transformed, 'toarray') else X_train_transformed
X_test_dense = X_test_transformed.toarray() if hasattr(X_test_transformed, 'toarray') else X_test_transformed

explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_dense,
    feature_names=all_feature_names,
    class_names=['Fail', 'Pass'],
    discretize_continuous=True
)

exp = explainer.explain_instance(
    X_test_dense[0],
    best_model.named_transformers_['classifier'].predict_proba,
    num_features=5
)
exp.save_to_file('results/lime_explanation.html')
print("Pipeline complete! Model and LIME HTML saved to 'results' folder.")
